In [17]:
#import
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [18]:
#load embeddings
embeddings = pd.read_csv("clinical_embeddings_16D.csv")

In [19]:
#define features & labels
embedding_cols = [f"embed_{i}" for i in range(1, 17)]
X = embeddings[embedding_cols].values
y = embeddings["BCR"].values
patient_ids = embeddings["patient_id"].values
embedding_fold = embeddings["fold"].values if "fold" in embeddings.columns else None

In [20]:
#5 fold cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [21]:
#define models
#logistic regression
log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
#random forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

models = {
    "logistic_regression": log_reg_model,
    "random_forest": rf_model
}

In [22]:
#run and generate summary
summary_rows = []

for model_name, model in models.items():
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #fit model
        model.fit(X_train, y_train)

        #predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        #predicted probabilities for class 1
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]

        #metrics for this fold
        fold_row = {
            "train_loss": log_loss(y_train, y_train_prob),
            "test_loss": log_loss(y_test, y_test_prob),
            "train_accuracy": accuracy_score(y_train, y_train_pred),
            "test_accuracy": accuracy_score(y_test, y_test_pred),
            "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_test_pred, zero_division=0)
        }
        fold_metrics.append(fold_row)

    #convert fold results to dataframe
    fold_metrics_df = pd.DataFrame(fold_metrics)

    #summarize across folds
    summary_row = {
        "model": model_name,
        "train_loss_mean": fold_metrics_df["train_loss"].mean(),
        "train_loss_sd": fold_metrics_df["train_loss"].std(),
        "test_loss_mean": fold_metrics_df["test_loss"].mean(),
        "test_loss_sd": fold_metrics_df["test_loss"].std(),
        "train_accuracy_mean": fold_metrics_df["train_accuracy"].mean(),
        "train_accuracy_sd": fold_metrics_df["train_accuracy"].std(),
        "test_accuracy_mean": fold_metrics_df["test_accuracy"].mean(),
        "test_accuracy_sd": fold_metrics_df["test_accuracy"].std(),
        "test_recall_mean": fold_metrics_df["test_recall"].mean(),
        "test_recall_sd": fold_metrics_df["test_recall"].std(),
        "test_f1_mean": fold_metrics_df["test_f1"].mean(),
        "test_f1_sd": fold_metrics_df["test_f1"].std()
    }

    summary_rows.append(summary_row)

#final summary
summary_df = pd.DataFrame(summary_rows)
print(summary_df)
summary_df.to_csv("clinical_embeddings_model_summary_stats.csv", index=False)

                 model  train_loss_mean  train_loss_sd  test_loss_mean  \
0  logistic_regression         0.301016       0.055947        1.033103   
1        random_forest         0.202961       0.026180        0.757663   

   test_loss_sd  train_accuracy_mean  train_accuracy_sd  test_accuracy_mean  \
0      0.438409               0.9375           0.044194                0.45   
1      0.168882               1.0000           0.000000                0.55   

   test_accuracy_sd  test_recall_mean  test_recall_sd  test_f1_mean  \
0          0.111803               0.4        0.223607          0.40   
1          0.325960               0.5        0.500000          0.46   

   test_f1_sd  
0    0.223607  
1    0.456070  


In [23]:
#load original clinical data
clinical_20 = pd.read_csv("subset_df_20.csv")
clinical_20.head()

,age_at_prostatectomy,ISUP,pre_operative_PSA,BCR,positive_surgical_margins,invasion_seminal_vesicles,lymphovascular_invasion,patient_id,pT_stage_2a,pT_stage_2c,pT_stage_3a,pT_stage_3b,pT_stage_4,pT_stage_4b,earlier_therapy_radiotherapy + cryotherapy,earlier_therapy_radiotherapy + hormones,earlier_therapy_unknown
0,63,3,24.0,1,0,1,1,1041,0,0,0,1,0,0,0,0,0
1,73,5,68.0,1,1,1,1,1090,0,0,0,1,0,0,0,0,0
2,65,4,7.9,1,0,0,1,1100,0,0,0,0,0,0,0,0,0
3,56,5,12.0,1,1,0,0,1122,0,0,0,0,1,0,0,0,0
4,64,3,7.8,1,1,0,0,1130,1,0,0,0,0,0,0,0,0


In [24]:
#define features & labels
target_col = "BCR"
id_col = "patient_id"

feature_cols = [col for col in clinical_20.columns if col not in [target_col, id_col]]
X2 = clinical_20[feature_cols].values
y2 = clinical_20[target_col].values
patient_ids = clinical_20[id_col].values

print("Feature columns:")
print(feature_cols)
print("\nX shape:", X.shape)
print("y shape:", y.shape)

Feature columns:
['age_at_prostatectomy', 'ISUP', 'pre_operative_PSA', 'positive_surgical_margins', 'invasion_seminal_vesicles', 'lymphovascular_invasion', 'pT_stage_2a', 'pT_stage_2c', 'pT_stage_3a', 'pT_stage_3b', 'pT_stage_4', 'pT_stage_4b', 'earlier_therapy_radiotherapy + cryotherapy', 'earlier_therapy_radiotherapy + hormones', 'earlier_therapy_unknown']

X shape: (20, 16)
y shape: (20,)


In [25]:
#5 fold cross validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [26]:
#define models
#logistic regression
log_reg_model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])

#random forest
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

models = {
    "logistic_regression": log_reg_model,
    "random_forest": rf_model
}

In [27]:
#run and generate summary
summary_rows = []
fold_results = []

for model_name, model in models.items():
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X2, y2), start=1):
        X_train, X_test = X2[train_idx], X2[test_idx]
        y_train, y_test = y2[train_idx], y2[test_idx]

        #fit model
        model.fit(X_train, y_train)

        #predictions
        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        #predicted probabilities for class 1
        y_train_prob = model.predict_proba(X_train)[:, 1]
        y_test_prob = model.predict_proba(X_test)[:, 1]

        #metrics for this fold
        fold_row = {
            "model": model_name,
            "fold": fold,
            "train_loss": log_loss(y_train, y_train_prob),
            "test_loss": log_loss(y_test, y_test_prob),
            "train_accuracy": accuracy_score(y_train, y_train_pred),
            "test_accuracy": accuracy_score(y_test, y_test_pred),
            "test_recall": recall_score(y_test, y_test_pred, zero_division=0),
            "test_f1": f1_score(y_test, y_test_pred, zero_division=0)
        }

        fold_metrics.append(fold_row)
        fold_results.append(fold_row)

    #convert fold results to dataframe
    fold_metrics_df = pd.DataFrame(fold_metrics)

    #summarize across folds
    summary_row = {
        "model": model_name,
        "train_loss_mean": fold_metrics_df["train_loss"].mean(),
        "train_loss_sd": fold_metrics_df["train_loss"].std(),
        "test_loss_mean": fold_metrics_df["test_loss"].mean(),
        "test_loss_sd": fold_metrics_df["test_loss"].std(),
        "train_accuracy_mean": fold_metrics_df["train_accuracy"].mean(),
        "train_accuracy_sd": fold_metrics_df["train_accuracy"].std(),
        "test_accuracy_mean": fold_metrics_df["test_accuracy"].mean(),
        "test_accuracy_sd": fold_metrics_df["test_accuracy"].std(),
        "test_recall_mean": fold_metrics_df["test_recall"].mean(),
        "test_recall_sd": fold_metrics_df["test_recall"].std(),
        "test_f1_mean": fold_metrics_df["test_f1"].mean(),
        "test_f1_sd": fold_metrics_df["test_f1"].std()
    }

    summary_rows.append(summary_row)

#final summary
summary_df = pd.DataFrame(summary_rows)
fold_results_df = pd.DataFrame(fold_results)

print("Summary across folds:")
print(summary_df)

summary_df.to_csv("clinical_model_summary_stats.csv", index=False)
fold_results_df.to_csv("clinical_model_fold_results.csv", index=False)

Summary across folds:
                 model  train_loss_mean  train_loss_sd  test_loss_mean  \
0  logistic_regression         0.213245       0.039994        0.416700   
1        random_forest         0.164941       0.016375        0.566068   

   test_loss_sd  train_accuracy_mean  train_accuracy_sd  test_accuracy_mean  \
0      0.144663                 0.95           0.027951                 0.8   
1      0.142600                 1.00           0.000000                 0.7   

   test_accuracy_sd  test_recall_mean  test_recall_sd  test_f1_mean  \
0          0.111803               0.8        0.273861      0.786667   
1          0.209165               0.7        0.273861      0.700000   

   test_f1_sd  
0    0.136626  
1    0.182574  


In [28]:
#fit random forest on full dataset to inspect feature importance
rf_model.fit(X2, y2)

feature_importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance_df

,feature,importance
2,pre_operative_PSA,0.221923
1,ISUP,0.174807
0,age_at_prostatectomy,0.170818
8,pT_stage_3a,0.102800
4,invasion_seminal_vesicles,0.098906
5,lymphovascular_invasion,0.060565
9,pT_stage_3b,0.041445
6,pT_stage_2a,0.037984
3,positive_surgical_margins,0.035919
7,pT_stage_2c,0.022229
